# Step 3: Calculate Evaluation Metrics - Template

This notebook template calculates comprehensive metrics from the LLM evaluations for any variant.
**Instructions**: Update the VARIANT variable below to specify which variant to analyze.

## L1 Metrics (Retrieval-Path Level)
- Violation Rate at K per retrieval path (lexical, kNN, hybrid)
- Precision@K per retrieval path
- UNCLEAR Rate per retrieval path

## L2 Metrics (End-to-End)
- Overall Violation Rate at K
- Overall Precision@K 
- Zero-Result Rate
- UNCLEAR Rate

## Breakdown Analysis
- Violation rates by negative intent type
- Search term patterns and examples
- Endpoint performance comparison

In [1]:
import pandas as pd
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Configuration

In [2]:
# Configuration - UPDATE VARIANT HERE
VARIANT = "control"  # Change to: "control", "treatment_v1", "treatment_v2", etc.

SEARCH_RESULTS_DIR = "./search_results"
MERGED_RESULTS_DIR = f"./llm_evaluations/merged_csv_results/{VARIANT}"
LLM_EVALUATIONS_DIR = "./llm_evaluations"
METRICS_OUTPUT_DIR = f"./metrics_output/{VARIANT}"
os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)

# Evaluation parameters
K_VALUES = [4, 10, 36]  # Different K values for Precision@K and Violation Rate@K
DEFAULT_K = 36  # Primary K value for reporting

print(f"Variant: {VARIANT.upper()}")
print(f"Merged results directory: {MERGED_RESULTS_DIR}")
print(f"LLM evaluations directory: {LLM_EVALUATIONS_DIR}")
print(f"Metrics output directory: {METRICS_OUTPUT_DIR}")
print(f"K values for evaluation: {K_VALUES}")
print(f"Default K: {DEFAULT_K}")

Variant: CONTROL
Merged results directory: ./llm_evaluations/merged_csv_results/control
LLM evaluations directory: ./llm_evaluations
Metrics output directory: ./metrics_output/control
K values for evaluation: [4, 10, 36]
Default K: 36


## Load Data

In [3]:
def load_merged_csv_results():
    """Load merged CSV results with LLM evaluations from step 2b"""
    
    all_results = []
    csv_files = []
    
    # Walk through merged results directory to find CSV files
    for root, dirs, files in os.walk(MERGED_RESULTS_DIR):
        for file in files:
            if file.endswith('.csv'):
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, MERGED_RESULTS_DIR)
                csv_files.append((full_path, rel_path))
    
    print(f"Loading {len(csv_files)} merged CSV files...")
    
    for full_path, rel_path in csv_files:
        try:
            df = pd.read_csv(full_path)
            
            # Extract endpoint type from filename
            filename = os.path.basename(rel_path)
            if filename.endswith('_search_results.csv'):
                endpoint_type = filename.replace('_search_results.csv', '')
            else:
                endpoint_type = os.path.splitext(filename)[0]  # fallback to filename without extension
            
            # Add metadata
            df['source_file'] = rel_path
            df['variant'] = VARIANT
            df['endpoint_type'] = endpoint_type
            
            # Filter for rows with LLM evaluations (negative intent queries)
            df_with_eval = df.dropna(subset=['llm_judgment']).copy()
            
            if len(df_with_eval) > 0:
                all_results.append(df_with_eval)
                print(f"  Loaded {len(df_with_eval)} evaluations from {endpoint_type} endpoint")
                    
        except Exception as e:
            print(f"Error loading {rel_path}: {e}")
            continue
    
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        print("Warning: No merged CSV files with LLM evaluations found")
        return pd.DataFrame()

def load_llm_evaluations():
    """Load LLM evaluation results for reference"""
    
    eval_file = os.path.join(LLM_EVALUATIONS_DIR, 'pair_judgments/final_evaluations.json')
    
    if not os.path.exists(eval_file):
        print(f"Warning: Evaluation file not found: {eval_file}")
        return pd.DataFrame()
    
    with open(eval_file, 'r') as f:
        evaluations = json.load(f)
    
    return pd.DataFrame(evaluations)

# Load merged data with evaluations
df_merged_results = load_merged_csv_results()
df_evaluations_ref = load_llm_evaluations()

print(f"\nMerged results with evaluations: {len(df_merged_results)} entries")
if len(df_evaluations_ref) > 0:
    print(f"Reference evaluations: {len(df_evaluations_ref)} entries")

if len(df_merged_results) > 0:
    print(f"\nColumns in merged data: {list(df_merged_results.columns)}")
    print(f"\n=== JUDGMENT DISTRIBUTION ===")
    judgment_counts = df_merged_results['llm_judgment'].value_counts()
    print(judgment_counts)
    print(f"\nPercentages:")
    for judgment, count in judgment_counts.items():
        print(f"  {judgment}: {count/len(df_merged_results)*100:.1f}%")
    
    print(f"\n=== VIOLATION RATE BY NEGATIVE INTENT ===")
    # Calculate violation rate by negative intent (excluding UNCLEAR)
    clear_judgments = df_merged_results[df_merged_results['llm_judgment'].isin(['VIOLATION', 'COMPLIANT'])]
    if len(clear_judgments) > 0:
        violation_by_intent = clear_judgments.groupby('negative_intent')['llm_judgment'].apply(
            lambda x: (x == 'VIOLATION').mean() * 100
        ).sort_values(ascending=False)
        print(violation_by_intent)
else:
    print("No data loaded. Please run step 2b first to generate merged CSV results.")

Loading 4 merged CSV files...
  Loaded 28465 evaluations from l1_hybrid endpoint
  Loaded 28465 evaluations from l2 endpoint
  Loaded 28609 evaluations from knn endpoint
  Loaded 26423 evaluations from lexical endpoint

Merged results with evaluations: 111962 entries
Reference evaluations: 67702 entries

Columns in merged data: ['search_term', 'endpoint_type', 'rank', 'doc_id', 'part_number', 'sku_name', 'score', 'negative_intent', 'llm_judgment', 'source_file', 'variant']

=== JUDGMENT DISTRIBUTION ===
llm_judgment
COMPLIANT    68834
VIOLATION    31181
UNCLEAR      11947
Name: count, dtype: int64

Percentages:
  COMPLIANT: 61.5%
  VIOLATION: 27.8%
  UNCLEAR: 10.7%

=== VIOLATION RATE BY NEGATIVE INTENT ===
negative_intent
kibble        100.000000
yeast         100.000000
grain, fat    100.000000
no bark       100.000000
bell          100.000000
                 ...    
tip             2.118644
gmo             1.700680
toxic           0.000000
iodine          0.000000
corn, soy       0

## Prepare Final Dataset

In [4]:
df_merged_results.head()

,search_term,endpoint_type,rank,doc_id,part_number,sku_name,score,negative_intent,llm_judgment,source_file,variant
0,2 hounds design freedom no pull dog harness,l1_hybrid,1,155034,127945,2 Hounds Design Freedom No Pull Nylon Dog Harn...,0.500000,pull,VIOLATION,l1_hybrid_search_results.csv,control
1,2 hounds design freedom no pull dog harness,l1_hybrid,2,148102,120925,"HDP Big Dog No Pull Dog Harness, Red, Large",0.333333,pull,VIOLATION,l1_hybrid_search_results.csv,control
2,2 hounds design freedom no pull dog harness,l1_hybrid,3,1098406,1098406,PetSafe Easy Walk Comfort Reflective No Pull D...,0.250000,pull,UNCLEAR,l1_hybrid_search_results.csv,control
3,2 hounds design freedom no pull dog harness,l1_hybrid,4,1074806,1074806,Joyride Harness Premium Advanced No Pull Dog H...,0.200000,pull,VIOLATION,l1_hybrid_search_results.csv,control
4,2 hounds design freedom no pull dog harness,l1_hybrid,5,304286,277885,"Frisco Basic No Pull Harness, Pink/Gray, XL",0.166667,pull,UNCLEAR,l1_hybrid_search_results.csv,control


In [5]:
# Use merged data directly - no need for separate merge step
df_final = df_merged_results.copy()

if len(df_final) > 0:
    print(f"\nFinal dataset: {len(df_final)} query-product pairs with evaluations")
    print(f"Unique queries: {df_final['search_term'].nunique()}")
    print(f"Unique products: {df_final['part_number'].nunique()}")
    print(f"Endpoint distribution:")
    print(df_final['endpoint_type'].value_counts())

    # Show sample of final data
    print(f"\nSample data:")
    sample_cols = ['search_term', 'endpoint_type', 'rank', 'sku_name', 'negative_intent', 'llm_judgment']
    available_cols = [col for col in sample_cols if col in df_final.columns]
    print(df_final[available_cols].head())
else:
    print("No data available for analysis")


Final dataset: 111962 query-product pairs with evaluations
Unique queries: 795
Unique products: 14032
Endpoint distribution:
endpoint_type
knn          28609
l1_hybrid    28465
l2           28465
lexical      26423
Name: count, dtype: int64

Sample data:
                                   search_term endpoint_type  rank  \
0  2 hounds design freedom no pull dog harness     l1_hybrid     1   
1  2 hounds design freedom no pull dog harness     l1_hybrid     2   
2  2 hounds design freedom no pull dog harness     l1_hybrid     3   
3  2 hounds design freedom no pull dog harness     l1_hybrid     4   
4  2 hounds design freedom no pull dog harness     l1_hybrid     5   

                                            sku_name negative_intent  \
0  2 Hounds Design Freedom No Pull Nylon Dog Harn...            pull   
1        HDP Big Dog No Pull Dog Harness, Red, Large            pull   
2  PetSafe Easy Walk Comfort Reflective No Pull D...            pull   
3  Joyride Harness Premium Advanced

## Search Term Analysis by Negative Intent

In [6]:
df_final.head()

,search_term,endpoint_type,rank,doc_id,part_number,sku_name,score,negative_intent,llm_judgment,source_file,variant
0,2 hounds design freedom no pull dog harness,l1_hybrid,1,155034,127945,2 Hounds Design Freedom No Pull Nylon Dog Harn...,0.500000,pull,VIOLATION,l1_hybrid_search_results.csv,control
1,2 hounds design freedom no pull dog harness,l1_hybrid,2,148102,120925,"HDP Big Dog No Pull Dog Harness, Red, Large",0.333333,pull,VIOLATION,l1_hybrid_search_results.csv,control
2,2 hounds design freedom no pull dog harness,l1_hybrid,3,1098406,1098406,PetSafe Easy Walk Comfort Reflective No Pull D...,0.250000,pull,UNCLEAR,l1_hybrid_search_results.csv,control
3,2 hounds design freedom no pull dog harness,l1_hybrid,4,1074806,1074806,Joyride Harness Premium Advanced No Pull Dog H...,0.200000,pull,VIOLATION,l1_hybrid_search_results.csv,control
4,2 hounds design freedom no pull dog harness,l1_hybrid,5,304286,277885,"Frisco Basic No Pull Harness, Pink/Gray, XL",0.166667,pull,UNCLEAR,l1_hybrid_search_results.csv,control


In [7]:
def analyze_search_terms_by_intent(df, k=DEFAULT_K):
    """Analyze search terms for each negative intent type"""
    
    if len(df) == 0:
        print("No data available for search term analysis")
        return
    
    # Filter to top K results
    top_k_data = df[df['rank'] <= k].copy()
    
    print(f"=== SEARCH TERM ANALYSIS BY NEGATIVE INTENT (K={k}) ===")
    
    for intent in df['negative_intent'].dropna().unique():
        intent_data = top_k_data[top_k_data['negative_intent'] == intent]
        
        if len(intent_data) == 0:
            continue
            
        print(f"\n--- NEGATIVE INTENT: '{intent}' ---")
        print(f"Total pairs: {len(intent_data)}")
        
        # Violation rate for this intent
        clear_judgments = intent_data[intent_data['llm_judgment'].isin(['VIOLATION', 'COMPLIANT'])]
        if len(clear_judgments) > 0:
            violation_rate = (clear_judgments['llm_judgment'] == 'VIOLATION').mean() * 100
            print(f"Violation rate: {violation_rate:.1f}%")
        
        # Top search terms for this intent
        top_queries = intent_data['search_term'].value_counts().head(5)
        print(f"\nTop search terms:")
        for query, count in top_queries.items():
            query_data = intent_data[intent_data['search_term'] == query]
            violations = (query_data['llm_judgment'] == 'VIOLATION').sum()
            total = len(query_data[query_data['llm_judgment'].isin(['VIOLATION', 'COMPLIANT'])])
            viol_rate = (violations / total * 100) if total > 0 else 0
            print(f"  '{query}' - {count} results, {violations}/{total} violations ({viol_rate:.1f}%)")
        
        # Examples of violations and compliant results
        violations = intent_data[intent_data['llm_judgment'] == 'VIOLATION']
        compliant = intent_data[intent_data['llm_judgment'] == 'COMPLIANT']
        
        if len(violations) > 0:
            print(f"\nExample VIOLATIONS:")
            for i, (idx, row) in enumerate(violations.head(2).iterrows()):
                product_name = row.get('sku_name', 'N/A')[:60]
                print(f"  {i+1}. '{row['search_term']}' → '{product_name}...'")
        
        if len(compliant) > 0:
            print(f"\nExample COMPLIANT:")
            for i, (idx, row) in enumerate(compliant.head(2).iterrows()):
                product_name = row.get('sku_name', 'N/A')[:60]
                print(f"  {i+1}. '{row['search_term']}' → '{product_name}...'")

# Run search term analysis
if len(df_final) > 0:
    analyze_search_terms_by_intent(df_final)
else:
    print("No data available for analysis")

=== SEARCH TERM ANALYSIS BY NEGATIVE INTENT (K=36) ===

--- NEGATIVE INTENT: 'pull' ---
Total pairs: 7259
Violation rate: 36.8%

Top search terms:
  'no pull collars' - 144 results, 68/143 violations (47.6%)
  'no pull collar' - 144 results, 98/141 violations (69.5%)
  'adjustable no pull harness' - 144 results, 132/135 violations (97.8%)
  'no pull dog collar' - 144 results, 47/139 violations (33.8%)
  'no pull dog harness' - 144 results, 22/144 violations (15.3%)

Example VIOLATIONS:
  1. '2 hounds design freedom no pull dog harness' → '2 Hounds Design Freedom No Pull Nylon Dog Harness & Leash, T...'
  2. '2 hounds design freedom no pull dog harness' → 'HDP Big Dog No Pull Dog Harness, Red, Large...'

Example COMPLIANT:
  1. '2 hounds design freedom no pull dog harness' → 'Gooby Escape Free Comfort X One Dog Harness, Portland Green,...'
  2. 'adjustable no pull harness' → 'Sporn Original No Pull Training Halter Dog Harness, Blue, La...'

--- NEGATIVE INTENT: 'grain' ---
Total pairs: 

## L1 Metrics: Retrieval-Path Level Analysis

In [8]:
def calculate_l1_metrics(df, k_values=K_VALUES):
    """Calculate L1 metrics per retrieval path - both per search term and aggregated"""
    
    if len(df) == 0:
        return {}
    
    l1_metrics = {}
    
    for endpoint in df['endpoint_type'].unique():
        endpoint_data = df[df['endpoint_type'] == endpoint].copy()
        endpoint_metrics = {}
        
        for k in k_values:
            # Filter to top K results per query
            top_k = endpoint_data[endpoint_data['rank'] <= k].copy()
            
            if len(top_k) == 0:
                continue
            
            # Calculate per-search-term metrics first
            search_term_metrics = []
            search_term_details = {}
            
            for search_term in top_k['search_term'].unique():
                term_data = top_k[top_k['search_term'] == search_term]
                
                # Calculate metrics for this search term
                total_results = len(term_data)
                violation_count = len(term_data[term_data['llm_judgment'] == 'VIOLATION'])
                compliant_count = len(term_data[term_data['llm_judgment'] == 'COMPLIANT'])
                unclear_count = len(term_data[term_data['llm_judgment'] == 'UNCLEAR'])
                
                term_violation_rate = violation_count / total_results if total_results > 0 else 0
                term_compliant_rate = compliant_count / total_results if total_results > 0 else 0
                term_unclear_rate = unclear_count / total_results if total_results > 0 else 0
                
                # Store individual search term details
                search_term_details[search_term] = {
                    f'violation_rate@{k}': term_violation_rate,
                    f'compliant_rate@{k}': term_compliant_rate,
                    f'unclear_rate@{k}': term_unclear_rate,
                    'total_results': total_results,
                    'violation_count': violation_count,
                    'compliant_count': compliant_count,
                    'unclear_count': unclear_count
                }
                
                # Collect for aggregation
                search_term_metrics.append({
                    f'violation_rate@{k}': term_violation_rate,
                    f'compliant_rate@{k}': term_compliant_rate,
                    f'unclear_rate@{k}': term_unclear_rate
                })
            
            # Calculate aggregated metrics (average across search terms)
            if search_term_metrics:
                agg_violation_rate = sum(m[f'violation_rate@{k}'] for m in search_term_metrics) / len(search_term_metrics)
                agg_compliant_rate = sum(m[f'compliant_rate@{k}'] for m in search_term_metrics) / len(search_term_metrics)
                agg_unclear_rate = sum(m[f'unclear_rate@{k}'] for m in search_term_metrics) / len(search_term_metrics)
            else:
                agg_violation_rate = agg_compliant_rate = agg_unclear_rate = None
            
            endpoint_metrics[f'K{k}'] = {
                # Aggregated metrics (average across search terms)
                f'violation_rate@{k}': agg_violation_rate,
                f'compliant_rate@{k}': agg_compliant_rate,
                f'unclear_rate@{k}': agg_unclear_rate,
                # Summary counts
                'total_search_terms': len(search_term_metrics),
                # Individual search term details
                'search_term_details': search_term_details
            }
        
        l1_metrics[endpoint] = endpoint_metrics
    
    return l1_metrics

# Calculate L1 metrics
if len(df_final) > 0:
    l1_metrics = calculate_l1_metrics(df_final)
    
    # Display L1 results
    print("\n=== L1 METRICS (Per Retrieval Path) ===")
    for endpoint, metrics in l1_metrics.items():
        print(f"\n{endpoint.upper()} ENDPOINT:")
        for k_level, k_metrics in metrics.items():
            k_val = k_level.replace('K', '')  # Extract K value
            print(f"\n  {k_level}:")
            print(f"    Violation Rate@{k_val}: {k_metrics[f'violation_rate@{k_val}']:.3f}, "
                  f"Compliant Rate@{k_val}: {k_metrics[f'compliant_rate@{k_val}']:.3f}, "
                  f"UNCLEAR Rate@{k_val}: {k_metrics[f'unclear_rate@{k_val}']:.3f}")
            print(f"    ({k_metrics['total_search_terms']} search terms)")
            
            # Show top/bottom search terms by violation rate
            term_details = k_metrics['search_term_details']
            sorted_terms = sorted(term_details.items(), key=lambda x: x[1][f'violation_rate@{k_val}'], reverse=True)
            
            print(f"\n  {k_level} - TOP 3 HIGHEST VIOLATION SEARCH TERMS:")
            for i, (term, details) in enumerate(sorted_terms[:3]):
                print(f"    {i+1}. '{term}': {details[f'violation_rate@{k_val}']:.3f} "
                      f"({details['violation_count']}/{details['total_results']})")
            
            print(f"\n  {k_level} - TOP 3 LOWEST VIOLATION SEARCH TERMS:")
            for i, (term, details) in enumerate(sorted_terms[-3:]):
                print(f"    {i+1}. '{term}': {details[f'violation_rate@{k_val}']:.3f} "
                      f"({details['violation_count']}/{details['total_results']})")


=== L1 METRICS (Per Retrieval Path) ===

L1_HYBRID ENDPOINT:

  K4:
    Violation Rate@4: 0.277, Compliant Rate@4: 0.665, UNCLEAR Rate@4: 0.058
    (795 search terms)

  K4 - TOP 3 HIGHEST VIOLATION SEARCH TERMS:
    1. 'beneful grain free': 1.000 (4/4)
    2. 'best no pull harness': 1.000 (4/4)
    3. 'blue buffalo chicken free': 1.000 (4/4)

  K4 - TOP 3 LOWEST VIOLATION SEARCH TERMS:
    1. 'wheat free puppy food': 0.000 (0/4)
    2. 'wholesomes grain free dog food': 0.000 (0/4)
    3. 'wild bird seed no waste': 0.000 (0/4)

  K10:
    Violation Rate@10: 0.281, Compliant Rate@10: 0.646, UNCLEAR Rate@10: 0.073
    (795 search terms)

  K10 - TOP 3 HIGHEST VIOLATION SEARCH TERMS:
    1. 'blue buffalo chicken free': 1.000 (10/10)
    2. 'cat collar no bell': 1.000 (10/10)
    3. 'cat litter no scent': 1.000 (10/10)

  K10 - TOP 3 LOWEST VIOLATION SEARCH TERMS:
    1. 'wheat free dog treats': 0.000 (0/10)
    2. 'wheat free puppy food': 0.000 (0/10)
    3. 'wild bird seed no waste': 0.

In [9]:
df_final.endpoint_type.unique()

array(['l1_hybrid', 'l2', 'knn', 'lexical'], dtype=object)

In [10]:
l1_metrics.keys()

dict_keys(['l1_hybrid', 'l2', 'knn', 'lexical'])

In [11]:
l1_metrics['l1_hybrid'].keys()

dict_keys(['K4', 'K10', 'K36'])

## L2 Metrics: End-to-End Analysis

In [12]:
# Extract L2 metrics from l1_metrics (already calculated for l2 endpoint)
l2_metrics = l1_metrics.get('l2', {})

if len(df_final) > 0 and len(l2_metrics) > 0:
    # Display L2 results
    print("\n=== L2 METRICS (End-to-End) ===")
    for k_level, k_metrics in l2_metrics.items():
        k_val = k_level.replace('K', '')  # Extract K value
        print(f"\n{k_level}:")
        print(f"  Violation Rate@{k_val}: {k_metrics[f'violation_rate@{k_val}']:.3f}")
        print(f"  Compliant Rate@{k_val}: {k_metrics[f'compliant_rate@{k_val}']:.3f}")
        print(f"  UNCLEAR Rate@{k_val}: {k_metrics[f'unclear_rate@{k_val}']:.3f}")
        print(f"  ({k_metrics['total_search_terms']} search terms)")
        
        # Show top/bottom search terms by violation rate
        term_details = k_metrics['search_term_details']
        sorted_terms = sorted(term_details.items(), key=lambda x: x[1][f'violation_rate@{k_val}'], reverse=True)
        
        print(f"\n{k_level} - TOP 5 HIGHEST VIOLATION SEARCH TERMS:")
        for i, (term, details) in enumerate(sorted_terms[:5]):
            print(f"  {i+1}. '{term}': {details[f'violation_rate@{k_val}']:.3f} "
                  f"({details['violation_count']}/{details['total_results']})")
        
        print(f"\n{k_level} - TOP 5 LOWEST VIOLATION SEARCH TERMS:")
        for i, (term, details) in enumerate(sorted_terms[-5:]):
            print(f"  {i+1}. '{term}': {details[f'violation_rate@{k_val}']:.3f} "
                  f"({details['violation_count']}/{details['total_results']})")
else:
    print("No L2 metrics available")


=== L2 METRICS (End-to-End) ===

K4:
  Violation Rate@4: 0.275
  Compliant Rate@4: 0.665
  UNCLEAR Rate@4: 0.060
  (795 search terms)

K4 - TOP 5 HIGHEST VIOLATION SEARCH TERMS:
  1. 'beneful grain free': 1.000 (4/4)
  2. 'best no pull harness': 1.000 (4/4)
  3. 'blue buffalo chicken free': 1.000 (4/4)
  4. 'cat collar no bell': 1.000 (4/4)
  5. 'cat food no chicken': 1.000 (4/4)

K4 - TOP 5 LOWEST VIOLATION SEARCH TERMS:
  1. 'wheat free dog food': 0.000 (0/4)
  2. 'wheat free dog treats': 0.000 (0/4)
  3. 'wheat free puppy food': 0.000 (0/4)
  4. 'wholesomes grain free dog food': 0.000 (0/4)
  5. 'wild bird seed no waste': 0.000 (0/4)

K10:
  Violation Rate@10: 0.278
  Compliant Rate@10: 0.650
  UNCLEAR Rate@10: 0.072
  (795 search terms)

K10 - TOP 5 HIGHEST VIOLATION SEARCH TERMS:
  1. 'blue buffalo chicken free': 1.000 (10/10)
  2. 'cat collar no bell': 1.000 (10/10)
  3. 'cat litter no scent': 1.000 (10/10)
  4. 'cat toys no catnip': 1.000 (10/10)
  5. 'cat toys without catnip':

## Breakdown Analysis by Negative Intent

In [13]:
def analyze_by_negative_intent(df, metrics_dict, k=DEFAULT_K):
    """Analyze metrics breakdown by negative intent - using pre-calculated search term metrics
    
    Args:
        df: Original dataframe for intent mapping
        metrics_dict: Dictionary with search_term_details (works with L1 endpoint metrics or L2 metrics)
        k: K value to analyze
    
    Returns:
        Dictionary mapping intent -> {aggregated metrics, search_term_details}
    """
    
    if len(df) == 0 or not metrics_dict:
        return {}
    
    # Get the search term details for this K value
    k_level = f'K{k}'
    if k_level not in metrics_dict:
        print(f"Warning: K{k} not found in metrics")
        return {}
    
    search_term_details = metrics_dict[k_level].get('search_term_details', {})
    
    # Map search terms to their negative intent
    df_top_k = df[df['rank'] <= k].copy()
    search_term_to_intent = df_top_k.drop_duplicates('search_term')[['search_term', 'negative_intent']].set_index('search_term')['negative_intent'].to_dict()
    
    intent_breakdown = {}
    
    # Group search term metrics by negative intent
    for search_term, metrics in search_term_details.items():
        intent = search_term_to_intent.get(search_term)
        if not intent or pd.isna(intent):
            continue
        
        if intent not in intent_breakdown:
            intent_breakdown[intent] = {
                'search_term_details': []
            }
        
        # Add search term with its metrics
        intent_breakdown[intent]['search_term_details'].append({
            'search_term': search_term,
            'violation_rate': metrics.get(f'violation_rate@{k}', 0),
            'compliant_rate': metrics.get(f'compliant_rate@{k}', 0),
            'unclear_rate': metrics.get(f'unclear_rate@{k}', 0),
            'total_results': metrics.get('total_results', 0),
            'violation_count': metrics.get('violation_count', 0),
            'compliant_count': metrics.get('compliant_count', 0),
            'unclear_count': metrics.get('unclear_count', 0)
        })
    
    # Calculate aggregated metrics for each intent
    for intent, data in intent_breakdown.items():
        search_terms = data['search_term_details']
        
        if search_terms:
            agg_violation_rate = sum(m['violation_rate'] for m in search_terms) / len(search_terms)
            agg_compliant_rate = sum(m['compliant_rate'] for m in search_terms) / len(search_terms)
            agg_unclear_rate = sum(m['unclear_rate'] for m in search_terms) / len(search_terms)
        else:
            agg_violation_rate = agg_compliant_rate = agg_unclear_rate = None
        
        data['aggregated'] = {
            'violation_rate': agg_violation_rate,
            'compliant_rate': agg_compliant_rate,
            'unclear_rate': agg_unclear_rate,
            'total_search_terms': len(search_terms)
        }
    
    return intent_breakdown

# Analyze by negative intent for L1 endpoints
all_endpoint_intent_breakdowns = {}
if len(df_final) > 0 and len(l1_metrics) > 0:
    for endpoint, endpoint_metrics in l1_metrics.items():
        endpoint_breakdown = analyze_by_negative_intent(df_final, endpoint_metrics)
        if endpoint_breakdown:
            all_endpoint_intent_breakdowns[endpoint] = endpoint_breakdown
            
            # Display L1 breakdown (skip l2 here, show separately below)
            if endpoint != 'l2':
                print(f"\n=== BREAKDOWN BY NEGATIVE INTENT - {endpoint.upper()} ENDPOINT (K={DEFAULT_K}) ===")
                for intent, metrics in endpoint_breakdown.items():
                    print(f"\n  --- {intent.upper()} ---")
                    agg = metrics['aggregated']
                    print(f"    Violation Rate: {agg['violation_rate']:.3f}, Compliant Rate: {agg['compliant_rate']:.3f}, UNCLEAR Rate: {agg['unclear_rate']:.3f}")
                    print(f"    ({agg['total_search_terms']} search terms)")
                    
                    # Show top 3 search terms by violation rate
                    search_terms = sorted(metrics['search_term_details'], key=lambda x: x['violation_rate'], reverse=True)
                    if search_terms:
                        print(f"    Top search terms:")
                        for i, st_metric in enumerate(search_terms[:3]):
                            print(f"      {i+1}. '{st_metric['search_term']}': {st_metric['violation_rate']:.3f} "
                                  f"({st_metric['violation_count']}/{st_metric['total_results']})")


=== BREAKDOWN BY NEGATIVE INTENT - L1_HYBRID ENDPOINT (K=36) ===

  --- PULL ---
    Violation Rate: 0.388, Compliant Rate: 0.588, UNCLEAR Rate: 0.024
    (53 search terms)
    Top search terms:
      1. 'dog harness medium no pull': 0.917 (33/36)
      2. 'sporn mesh no pull dog harness': 0.913 (21/23)
      3. 'freedom no pull harness': 0.889 (32/36)

  --- GRAIN ---
    Violation Rate: 0.194, Compliant Rate: 0.755, UNCLEAR Rate: 0.051
    (332 search terms)
    Top search terms:
      1. 'hills science diet grain free': 0.944 (34/36)
      2. 'hills science grain free': 0.917 (33/36)
      3. 'science diet grain free': 0.833 (30/36)

  --- ALLER ---
    Violation Rate: 0.306, Compliant Rate: 0.611, UNCLEAR Rate: 0.083
    (1 search terms)
    Top search terms:
      1. 'aller free': 0.306 (11/36)

  --- ALLERGY ---
    Violation Rate: 0.241, Compliant Rate: 0.704, UNCLEAR Rate: 0.056
    (3 search terms)
    Top search terms:
      1. 'allergy free dog treats': 0.306 (11/36)
      

In [14]:
# Analyze by negative intent for L2 (end-to-end)
l2_intent_breakdown = {}
if len(df_final) > 0 and len(l2_metrics) > 0:
    l2_intent_breakdown = analyze_by_negative_intent(df_final, l2_metrics)
    
    print(f"\n=== BREAKDOWN BY NEGATIVE INTENT - L2 (END-TO-END) (K={DEFAULT_K}) ===")
    for intent, metrics in l2_intent_breakdown.items():
        print(f"\n  --- {intent.upper()} ---")
        agg = metrics['aggregated']
        print(f"    Violation Rate: {agg['violation_rate']:.3f}, Compliant Rate: {agg['compliant_rate']:.3f}, UNCLEAR Rate: {agg['unclear_rate']:.3f}")
        print(f"    ({agg['total_search_terms']} search terms)")
        
        # Show top 5 search terms by violation rate
        search_terms = sorted(metrics['search_term_details'], key=lambda x: x['violation_rate'], reverse=True)
        if search_terms:
            print(f"    Top search terms:")
            for i, st_metric in enumerate(search_terms[:5]):
                print(f"      {i+1}. '{st_metric['search_term']}': {st_metric['violation_rate']:.3f} "
                      f"({st_metric['violation_count']}/{st_metric['total_results']})")


=== BREAKDOWN BY NEGATIVE INTENT - L2 (END-TO-END) (K=36) ===

  --- PULL ---
    Violation Rate: 0.386, Compliant Rate: 0.589, UNCLEAR Rate: 0.024
    (53 search terms)
    Top search terms:
      1. 'sporn mesh no pull dog harness': 0.913 (21/23)
      2. 'dog harness medium no pull': 0.889 (32/36)
      3. 'freedom no pull harness': 0.889 (32/36)
      4. 'adjustable no pull harness': 0.861 (31/36)
      5. 'no slip harness': 0.861 (31/36)

  --- GRAIN ---
    Violation Rate: 0.192, Compliant Rate: 0.757, UNCLEAR Rate: 0.051
    (332 search terms)
    Top search terms:
      1. 'hills science diet grain free': 0.944 (34/36)
      2. 'hills science grain free': 0.917 (33/36)
      3. 'science diet grain free': 0.833 (30/36)
      4. 'purina pro plan grain free': 0.778 (28/36)
      5. 'purina one grain free dog food': 0.750 (27/36)

  --- ALLER ---
    Violation Rate: 0.306, Compliant Rate: 0.611, UNCLEAR Rate: 0.083
    (1 search terms)
    Top search terms:
      1. 'aller free': 

## Export Metrics to CSV

In [15]:
# Export metrics: 2 simple CSVs covering all endpoints (lexical, knn, l1_hybrid, l2)

if not l1_metrics:
    print("No metrics available to export")
else:
    print(f"\n=== EXPORTING METRICS ===")
    
    # 1. AGGREGATED METRICS (averaged across all search terms)
    aggregated_data = []
    for endpoint, endpoint_metrics in l1_metrics.items():
        for k_level, k_metrics in endpoint_metrics.items():
            k_val = k_level.replace('K', '')
            aggregated_data.append({
                'variant': VARIANT,
                'endpoint': endpoint,
                'k': k_val,
                'violation_rate': k_metrics[f'violation_rate@{k_val}'],
                'compliant_rate': k_metrics[f'compliant_rate@{k_val}'],
                'unclear_rate': k_metrics[f'unclear_rate@{k_val}'],
                'total_search_terms': k_metrics['total_search_terms']
            })
    
    aggregated_df = pd.DataFrame(aggregated_data)
    aggregated_file = os.path.join(METRICS_OUTPUT_DIR, f'{VARIANT}_aggregated_metrics.csv')
    aggregated_df.to_csv(aggregated_file, index=False)
    print(f"✓ Aggregated metrics saved: {aggregated_file}")
    print(f"  Shape: {aggregated_df.shape} (rows × columns)")
    
    # 2. SEARCH TERM LEVEL METRICS (detailed per search term, per endpoint)
    search_term_data = []
    for endpoint, endpoint_metrics in l1_metrics.items():
        for k_level, k_metrics in endpoint_metrics.items():
            if 'search_term_details' in k_metrics:
                k_val = k_level.replace('K', '')
                for search_term, details in k_metrics['search_term_details'].items():
                    search_term_data.append({
                        'variant': VARIANT,
                        'endpoint': endpoint,
                        'k': k_val,
                        'search_term': search_term,
                        'violation_rate': details[f'violation_rate@{k_val}'],
                        'compliant_rate': details[f'compliant_rate@{k_val}'],
                        'unclear_rate': details[f'unclear_rate@{k_val}'],
                        'total_results': details['total_results'],
                        'violation_count': details['violation_count'],
                        'compliant_count': details['compliant_count'],
                        'unclear_count': details['unclear_count']
                    })
    
    search_term_df = pd.DataFrame(search_term_data)

    # Add negative_intent column from df_final
    if len(df_final) > 0 and 'negative_intent' in df_final.columns:
        search_term_to_intent = df_final.drop_duplicates('search_term')[['search_term', 'negative_intent']].set_index('search_term')['negative_intent'].to_dict()
        search_term_df['negative_intent'] = search_term_df['search_term'].map(search_term_to_intent)
        intent_coverage = search_term_df['negative_intent'].notna().sum()
        print(f"Added negative_intent: {intent_coverage}/{len(search_term_df)} rows have intent")
    else:
        search_term_df['negative_intent'] = None
        print("negative_intent column added (all null)")

    # Attach search_volume from candidates file using strict mapping (Q -> search_term, CNT -> search_volume)
    candidates_file = "./negative_intent_candidates(raw_data).csv"
    if os.path.exists(candidates_file):
        try:
            df_cand_raw = pd.read_csv(candidates_file)
            df_cand = df_cand_raw[['Q', 'CNT']].rename(columns={'Q': 'search_term', 'CNT': 'search_volume'})
            df_cand['search_volume'] = pd.to_numeric(df_cand['search_volume'], errors='coerce').fillna(0)
            df_cand = df_cand.sort_values('search_volume', ascending=False).drop_duplicates('search_term')
            before = len(search_term_df)
            search_term_df = search_term_df.merge(df_cand, on='search_term', how='left')
            attached = search_term_df['search_volume'].notna().sum()
            print(f"Attached search_volume for {attached}/{before} search-term rows")
        except Exception as e:
            print(f"Error merging search_volume: {e}")
    else:
        print("Candidates file not found; exporting without search_volume")
    
    search_term_file = os.path.join(METRICS_OUTPUT_DIR, f'{VARIANT}_search_term_metrics.csv')
    search_term_df.to_csv(search_term_file, index=False)
    print(f"✓ Search term metrics saved: {search_term_file}")
    print(f"  Shape: {search_term_df.shape} (rows × columns)")
    
    print(f"\n=== EXPORT COMPLETE ===")
    print(f"All files saved to: {METRICS_OUTPUT_DIR}")


=== EXPORTING METRICS ===
✓ Aggregated metrics saved: ./metrics_output/control/control_aggregated_metrics.csv
  Shape: (12, 7) (rows × columns)
Added negative_intent: 9534/9534 rows have intent
Attached search_volume for 9534/9534 search-term rows
✓ Search term metrics saved: ./metrics_output/control/control_search_term_metrics.csv
  Shape: (9534, 13) (rows × columns)

=== EXPORT COMPLETE ===
All files saved to: ./metrics_output/control


In [16]:
search_term_df.negative_intent.unique()

array(['pull', 'grain', 'aller', 'allergy', 'waste free', 'shock', 'odor',
       'mess', 'waste', 'melt', 'chicken', 'bpa', 'd3', 'hide', 'bell',
       'prescription', 'additives', 'dust', 'fragrance', 'scent',
       'tracking', 'pee', 'scratch', 'catnip', 'carpet', 'rawhide', 'poo',
       'poop', 'medicated', 'non gmo', 'soy', 'choke', 'sunflower seeds',
       'corn and soy', 'corn', 'free', 'seafood', 'poultry', 'escape',
       'zip', 'squeak', 'squeaker', 'stuffing', 'teeth', 'spill', 'fat',
       'fish', 'ears', 'non clumping', 'gain', 'gluten', 'gmo',
       'grain, chicken', 'probiotic', 'iodine', 'clumping', 'legume',
       'pill', 'grain, fat', 'bark', 'no bark', 'cat pee', 'chew',
       'clump', 'grow', 'hides', 'holes', 'lick', 'no melt', 'millet',
       'poop eating', 'scoot', 'spray', 'squirrel', 'sugar', 'tip',
       'toot', 'track', 'non absorbent', 'clay',
       'non clumping unscented cat litter\n\nthe user is actually looking for cat litter',
       'kibble

In [17]:
len(search_term_df.negative_intent.unique())

102